# Model Architecture Visualization

Visualize any project model with multiple backends:
**torchviz** (computational graph), **torchsummary** (text summary),
**nnviz** (fx graph), **torchlens** (activation graph), **visualtorch** (layered diagram).

In [ ]:
from __future__ import annotations

import os
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch
import torch.nn as nn
from IPython.display import SVG, Image, display

sys.path.insert(0, os.path.abspath(".."))
from utils import get_model_from_config

warnings.filterwarnings("ignore", category=UserWarning)

# ---------------------------------------------------------------------------
# Model registry – every model that exists locally with its default config
# ---------------------------------------------------------------------------

@dataclass
class ModelEntry:
    slug: str
    display_name: str
    model_type: str
    config_path: str

MODELS: dict[str, ModelEntry] = {}

def _reg(slug: str, display_name: str, model_type: str, config_path: str):
    MODELS[slug] = ModelEntry(slug, display_name, model_type, config_path)

_reg("edge_bs_roformer",   "Edge-BS-RoFormer (64)",          "edge_bs_rof",       "configs/3_FA_RoPE(64).yaml")
_reg("edge_bs_roformer_s", "Edge-BS-RoFormer (48)",          "edge_bs_rof",       "configs/3_FA_RoPE(48).yaml")
_reg("dcunet",             "DCUNet",                         "dcunet",            "configs/5_Baseline_dcunet.yaml")
_reg("dcunet_rps_bn",      "DCUNet + RPS (bottleneck)",      "dcunet",            "configs/5a_DCUNet_RPS_bottleneck.yaml")
_reg("dcunet_rps_gru",     "DCUNet + RPS (GRU)",             "dcunet",            "configs/5b_DCUNet_RPS_gru.yaml")
_reg("dcunet_rps_hier",    "DCUNet + RPS (hierarchical)",    "dcunet",            "configs/5c_DCUNet_RPS_hierarchical.yaml")
_reg("dptnet",             "DPTNet",                         "dptnet",            "configs/7_Baseline_dptnet.yaml")
_reg("htdemucs",           "HTDemucs",                       "htdemucs",          "configs/8_Baseline_htdemucs.yaml")
_reg("diffusion_buffer",   "Diffusion Buffer (BBED)",        "diffusion_buffer",  "configs/9_Diffusion_Buffer_BBED.yaml")
_reg("dcunet_dregon",      "DCUNet (DREGON)",                "dcunet",            "configs/7b_DCUNet_baseline_DREGON.yaml")
_reg("dcunet_dregon_rps",  "DCUNet + RPS (DREGON)",          "dcunet",            "configs/7a_DCUNet_RPS_DREGON.yaml")
_reg("dcunet_dregon_rps_bn",  "DCUNet + RPS bn (DREGON)",   "dcunet",            "configs/6a_DCUNet_RPS_DREGON_bottleneck.yaml")
_reg("dcunet_dregon_rps_gru", "DCUNet + RPS GRU (DREGON)",  "dcunet",            "configs/6b_DCUNet_RPS_DREGON_gru.yaml")
_reg("dcunet_dregon_rps_hier","DCUNet + RPS hier (DREGON)",  "dcunet",            "configs/6c_DCUNet_RPS_DREGON_hierarchical.yaml")
_reg("dccrn",              "DCCRN",                          "dccrn",             "configs/10a_DCCRN_baseline_DREGON.yaml")
_reg("dccrn_rps",          "DCCRN + RPS",                    "dccrn",             "configs/10b_DCCRN_RPS_DREGON.yaml")
_reg("dccrn_lite_rps",     "DCCRN-Lite + RPS",               "dccrn",             "configs/10c_DCCRNLite_RPS_DREGON.yaml")
_reg("dccrn_rps_pred",     "DCCRN + RPS + PredRPS",          "dccrn",             "configs/10d_DCCRN_RPS_PredRPS_DREGON.yaml")

print("Available model slugs:")
for slug, entry in MODELS.items():
    print(f"  {slug:25s}  {entry.display_name}")

In [ ]:
Backend = Literal["torchviz", "torchsummary", "nnviz", "torchlens", "visualtorch"]

ALL_BACKENDS: list[Backend] = ["torchviz", "torchsummary", "nnviz", "torchlens", "visualtorch"]


def _make_dummy_input(model_type: str, config) -> tuple[torch.Tensor, dict]:
    """Create a dummy waveform tensor matching the model's expected input."""
    if model_type == "htdemucs":
        sr = getattr(config.training, "samplerate", 16000)
        channels = getattr(config.training, "channels", 1)
        segment = getattr(config.training, "segment", 1)
        length = int(sr * segment)
        return torch.randn(1, channels, length), {}

    chunk_size = int(config.audio.get("chunk_size", 16000))
    num_channels = int(config.audio.get("num_channels", 1))
    return torch.randn(1, num_channels, chunk_size), {}


def _patch_flash_attn_for_cpu(model: nn.Module):
    """Disable flash attention flags on Attend modules so the model runs on CPU."""
    from models.edge_bs_rof.attend import Attend
    for m in model.modules():
        if isinstance(m, Attend):
            m.flash = False


def _load_model(slug: str, device: str = "cpu") -> tuple[nn.Module, str, object, torch.Tensor, dict]:
    """Load model and produce a matching dummy input."""
    entry = MODELS[slug]
    model, config = get_model_from_config(entry.model_type, entry.config_path)
    model = model.to(device).eval()
    if device == "cpu":
        _patch_flash_attn_for_cpu(model)
    dummy, extra_kw = _make_dummy_input(entry.model_type, config)
    dummy = dummy.to(device)
    return model, entry.model_type, config, dummy, extra_kw


# ---------------------------------------------------------------------------
# Individual backend wrappers
# ---------------------------------------------------------------------------

def _viz_torchviz(model, dummy, extra_kw, slug, save_path: Path | None):
    """Computational graph via torchviz (Graphviz)."""
    from torchviz import make_dot

    out = model(dummy, **extra_kw)
    if isinstance(out, tuple):
        out = out[0]
    dot = make_dot(out, params=dict(model.named_parameters()), show_attrs=False, show_saved=False)
    dot.attr(rankdir="TB")
    if save_path:
        dot.render(str(save_path.with_suffix("")), format="png", cleanup=True)
        display(Image(filename=str(save_path.with_suffix(".png"))))
    else:
        display(SVG(dot.pipe(format="svg")))


def _viz_torchsummary(model, dummy, extra_kw, slug, save_path: Path | None):
    """Text summary table (layer shapes + param counts)."""
    from torchsummary import summary

    input_size = tuple(dummy.shape[1:])
    summary(model, input_size=input_size, device=str(dummy.device))


def _viz_nnviz(model, dummy, extra_kw, slug, save_path: Path | None):
    """
    torch.fx-based graph via nnviz.
    NOTE: fails on models with dynamic control flow (if/for on tensor values).
    """
    from nnviz.drawing.graphviz import GraphvizDrawer
    from nnviz.inspection.torchfx import TorchFxInspector

    inspector = TorchFxInspector()
    graph = inspector.inspect(model, dummy)
    drawer = GraphvizDrawer()
    out_path = str(save_path.with_suffix(".png")) if save_path else f"/tmp/nnviz_{slug}.png"
    drawer.draw(graph, out_path)
    display(Image(filename=out_path))


def _viz_torchlens(model, dummy, extra_kw, slug, save_path: Path | None):
    """Full computational graph with activation metadata via TorchLens."""
    import torchlens as tl

    out_path = str(save_path.with_suffix("")) if save_path else f"/tmp/torchlens_{slug}"
    model_log = tl.log_forward_pass(
        model, dummy,
        layers_to_save="none",
        vis_mode="unrolled",
        vis_outpath=out_path,
        vis_fileformat="png",
        vis_save_only=True,
    )
    png_path = out_path + ".png"
    if os.path.exists(png_path):
        display(Image(filename=png_path))
    else:
        print(f"  [torchlens] rendered to {out_path}.*  (check file format)")


def _viz_visualtorch(model, dummy, extra_kw, slug, save_path: Path | None):
    """
    Layered / graph diagram via visualtorch (PIL image).
    Works best on Sequential-style models; may be slow on very deep graphs.
    """
    import visualtorch

    input_shape = list(dummy.shape[1:])
    img = visualtorch.graph_view(model, input_shape=input_shape)
    if save_path:
        img.save(str(save_path.with_suffix(".png")))
    display(img)


_DISPATCH: dict[Backend, callable] = {
    "torchviz": _viz_torchviz,
    "torchsummary": _viz_torchsummary,
    "nnviz": _viz_nnviz,
    "torchlens": _viz_torchlens,
    "visualtorch": _viz_visualtorch,
}

BACKEND_NOTES: dict[Backend, str] = {
    "torchviz":     "Computational graph (data-flow DAG). Works on any model.",
    "torchsummary": "Text table of layers, output shapes, param counts. Always works.",
    "nnviz":        "torch.fx graph. Fails on dynamic control flow (if/for on tensors).",
    "torchlens":    "Full op graph with activation metadata. Works on any model.",
    "visualtorch":  "Pretty layered diagram (PIL). Can be slow on non-Sequential models.",
}


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def visualize(
    slug: str,
    backend: Backend = "torchsummary",
    device: str = "cpu",
    save_dir: str | Path | None = None,
):
    """
    Visualize a model architecture.

    Parameters
    ----------
    slug : str
        Model identifier (see ``MODELS`` dict).
    backend : str
        One of: torchviz, torchsummary, nnviz, torchlens, visualtorch.
    device : str
        ``'cpu'`` or ``'cuda'``.  CPU is safest for visualization.
    save_dir : path, optional
        If given, renders are saved as ``<slug>_<backend>.png``.
    """
    if slug not in MODELS:
        raise KeyError(f"Unknown slug {slug!r}. Choose from:\n  " + "\n  ".join(MODELS))
    if backend not in _DISPATCH:
        raise ValueError(f"Unknown backend {backend!r}. Choose from: {list(_DISPATCH)}")

    entry = MODELS[slug]
    print(f"{'=' * 60}")
    print(f"  {entry.display_name}  [{slug}]")
    print(f"  backend = {backend}")
    print(f"  {BACKEND_NOTES[backend]}")
    print(f"{'=' * 60}")

    model, model_type, config, dummy, extra_kw = _load_model(slug, device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters: {n_params:,}")

    save_path = None
    if save_dir:
        save_path = Path(save_dir) / f"{slug}_{backend}"
        save_path.parent.mkdir(parents=True, exist_ok=True)

    fn = _DISPATCH[backend]
    try:
        fn(model, dummy, extra_kw, slug, save_path)
    except Exception as exc:
        print(f"\n  [{backend}] failed for {slug}: {exc}")
        import traceback; traceback.print_exc()


def visualize_all_backends(slug: str, device: str = "cpu", save_dir: str | Path | None = None):
    """Run every available backend on a single model."""
    for b in ALL_BACKENDS:
        visualize(slug, backend=b, device=device, save_dir=save_dir)


def list_backends():
    """Print available backends with descriptions."""
    for name, note in BACKEND_NOTES.items():
        print(f"  {name:15s}  {note}")


print("Ready!  Use:\n"
      "  visualize(slug, backend)          — single visualization\n"
      "  visualize_all_backends(slug)      — all backends on one model\n"
      "  list_backends()                   — show backend descriptions")

## Quick start

```python
# Pick a model slug and a backend:
visualize("dcunet", backend="torchsummary")
visualize("edge_bs_roformer", backend="torchviz")

# Try all backends on one model:
visualize_all_backends("dptnet")

# Save renders to disk:
visualize("dccrn", backend="nnviz", save_dir="outputs/viz")
```

In [ ]:
# --- Edit these two values and run ---
SLUG = "dcunet"
BACKEND: Backend = "torchsummary"

visualize(SLUG, backend=BACKEND)